# 04 — Dead-ball and set-piece boundary study

Smart Tagging opens a restart review interval; it is not the exact kick frame. This notebook studies the physical boundary between D and live H/A evidence. `first_half_gt.csv` is used only as a retrospective, hand-annotated insight source.

George's `p_dead`, source, speed, and uncertainty metadata can support a missing-ball dead-ball hypothesis. Haiqal's dead-ball score is another bounded observation channel. For ordinary set pieces, restart geometry, physical ball activation, and subsequent controlled movement remain necessary before releasing D. Kickoff is the explicit exception: the pre-kickoff interval is D, then the first physically active frame is immediately anchored to the kickoff team.

Restart-boundary diagrams, including both half-start kickoff examples, belong with the anchor audit in notebook 05; this notebook keeps the dead-ball/set-piece summary table.

## Evidence chain

- Restart type: Smart Tag context, not possession truth.
- Restart activation: near-zero ball state followed by active ball movement.
- Kick outcome: ball speed, acceleration, direction, and distance per kick.
- Receiver evidence: trajectory intersection, landing-area density, approach speed, and later control.
- Goal aftermath: preserve D until a centre-circle restart is physically supported.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd().resolve()
while not (ROOT / 'possession_lab').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from possession_lab.calculations.features import build_engineered_features
from possession_lab.calculations.plotting import plot_timeline_window
from possession_lab.baseline_pipeline import ClosestTeamRuleModel
from starter.load_smart_tagging import load as load_smart_tagging
features = build_engineered_features(model=ClosestTeamRuleModel())
gt_path = ROOT / 'data' / 'first_half_gt.csv'
insight_gt = pd.read_csv(gt_path).set_index('frame')['label'] if gt_path.exists() else pd.Series(dtype=object)
tags = pd.DataFrame(load_smart_tagging())
print('Smart Tag events:', len(tags), '| retrospective GT:', not insight_gt.empty)

Smart Tag events: 638 | retrospective GT: True


## Restart-specific interpretation table

| Restart | What we inspect | Conservative handling |
|---|---|---|
| Goal kick | goal-area restart context, near-zero state, kick acceleration, short/long receiver evidence | retain D until live kick and receiver/control evidence agree |
| Free kick | Smart Tag context, stationary-to-active change, pass or flight direction | do not treat the tag end as H/A; inspect the first controlled outcome |
| Corner | proximity to the actual pitch corner, activation, landing density and player convergence | hold D through delivery/contest; transition only after control |
| Throw-in | touchline context, expected missingness, release acceleration and first receiver | tolerate missing ball; use first stable receiver, not fixed duration |
| Kickoff / after goal | centre-circle restart, near-zero-to-active movement | keep pre-kickoff as D; assign the first active frame immediately to the tagged kickoff team |

In [2]:
ball_missing = features['ball_x'].isna() | features['ball_y'].isna()
rows = []
for _, event in tags.iterrows():
    start = int(event['start_frame']); end = int(event['end_frame'])
    part = features.loc[features['frame'].between(start, end)]
    if len(part):
        event_tags = event.get('tags', {}) if isinstance(event.get('tags', {}), dict) else {}
        rows.append({'type': event_tags.get('Type', event.get('Type', event.get('type', 'unknown'))), 'start_frame': start, 'end_frame': end, 'missing_share': float(ball_missing.loc[part.index].mean()), 'frames': len(part)})
audit = pd.DataFrame(rows)
display(audit.groupby('type').agg(events=('type','size'), mean_missing_share=('missing_share','mean'), median_frames=('frames','median')).sort_index())

,events,mean_missing_share,median_frames
type,,,
Corner Kick,12,0.408295,876.0
Free Kick,5,0.405936,876.0
Goal Kick,10,0.266781,876.0
Kick Off (After Goal),4,0.726190,126.0
Kick Off (Start),2,0.357143,126.0
Throw-in,26,0.253952,876.0
unknown,579,0.229775,552.0


In [3]:
# Restart diagrams are kept with the anchor audit in notebook 05; this notebook
# remains focused on dead-ball/set-piece summary statistics.
display(audit.sort_values(['type', 'start_frame']).reset_index(drop=True))


,type,start_frame,end_frame,missing_share,frames
0,Corner Kick,29608,30483,0.561644,876
1,Corner Kick,34651,35526,0.452055,876
2,Corner Kick,36939,37814,0.128995,876
3,Corner Kick,42614,43489,0.310502,876
4,Corner Kick,57368,58243,0.374429,876
...,...,...,...,...,...
633,unknown,187713,188024,0.000000,312
634,unknown,187752,188161,0.000000,410
635,unknown,188666,189236,0.000000,571
636,unknown,188666,189236,0.000000,571


## Pipeline conclusion

- Smart Tagging identifies where to inspect.
- Physical restart activation identifies when live play begins.
- Kick distance, speed, acceleration, direction, density, and receiver control determine confidence.
- The decoder receives the same evidence for every scenario and decides how conservatively to transition from D to H/A.